# Module 07 - Capstone

**Duration:** 30 minutes

This is your session to put it all together with your own material.
The notebook gives you a structure to follow, but you make the decisions.

At the end, each person or pair shares one finding with the group (1 minute).

---


## Step 1 - Choose your documents

Put one or more documents into `data/my_docs/`.
They can be anything: notes, articles, a paper you are reading, a contract, a manual.

Supported formats: `.txt`, `.pdf`, `.docx`

If you do not have documents with you, use the existing sample docs
but focus your questions on a specific theme.


In [ ]:
import os

MY_DATA_PATH = '../data/my_docs'
MY_COLLECTION = 'my_collection'

os.makedirs(MY_DATA_PATH, exist_ok=True)

files = [f for f in os.listdir(MY_DATA_PATH) if not f.startswith('.')]
if not files:
    print('No files found in data/my_docs/')
    print('Add at least one .txt, .pdf, or .docx file and re-run this cell.')
else:
    print(f'Found {len(files)} file(s):')
    for f in files:
        print(f'  {f}')


## Step 2 - Ingest


In [ ]:
from ragsst.ragtool import RAGTool

tool = RAGTool(data_path=MY_DATA_PATH, collection_name=MY_COLLECTION)
tool.make_collection(MY_DATA_PATH, MY_COLLECTION)

print(f'Collection ready: {tool.collection.count()} chunks')


## Step 3 - Run a baseline

Write 3 to 5 questions that you expect your documents to answer.
Run them through the basic pipeline and check the results.


In [ ]:
import json
from os import getenv
from urllib.parse import urljoin

import requests

OLLAMA_URL = urljoin(getenv('OLLAMA_HOST', 'http://localhost:11434'), 'api')


def generate(prompt: str, temp: float = 0.3) -> str:
    r = requests.post(
        OLLAMA_URL + '/generate',
        json={'model': 'llama3.2:3b', 'prompt': prompt, 'stream': False,
              'options': {'temperature': temp}}
    )
    return json.loads(r.text).get('response', '')


# TODO: replace these with questions relevant to your documents
my_questions = [
    'Question 1',
    'Question 2',
    'Question 3',
]

print('BASELINE RESULTS')
print('=' * 60)
for q in my_questions:
    context = tool.get_relevant_text(q, nresults=3)
    answer = generate(tool.get_context_prompt(q, context))
    print(f'Q: {q}')
    print(f'A: {answer[:300]}')
    print()


## Step 4 - Apply one improvement technique

Pick one technique from Module 05 and apply it to at least one of your questions.

Choose based on what you observed in Step 3:
- If the retriever found the wrong chunk, try re-ranking or multi-query.
- If the questions are short and the documents use different vocabulary, try HyDE.
- If the answers are repetitive or shallow, try increasing nresults.

Copy the relevant code from Module 05 into the cell below.


In [ ]:
# TODO: implement your chosen technique here
# Then run the same questions from Step 3 and compare

print('IMPROVED RESULTS')
print('=' * 60)


## Step 5 - Measure

Use the evaluation tools from Module 06 to measure whether your improvement actually helped.

You need at least 3 questions with known expected answers to get a meaningful score.


In [ ]:
# Build a mini gold dataset for your documents
# expected_fragment: a word or phrase that should appear in a correct answer

my_gold = [
    {'question': 'Question 1', 'expected_fragment': 'expected word or phrase', 'source': 'yourfile.txt'},
    {'question': 'Question 2', 'expected_fragment': 'expected word or phrase', 'source': 'yourfile.txt'},
    {'question': 'Question 3', 'expected_fragment': 'expected word or phrase', 'source': 'yourfile.txt'},
]


def evaluate(dataset, k=3):
    results = []
    for item in dataset:
        context = tool.get_relevant_text(item['question'], nresults=k)
        answer = generate(tool.get_context_prompt(item['question'], context))
        retrieval = tool.collection.query(
            query_texts=[item['question']], n_results=k, include=['metadatas']
        )
        sources = [m['source'] for m in retrieval['metadatas'][0]]
        results.append({
            'q': item['question'],
            'retrieval': item['source'] in sources,
            'answer': item['expected_fragment'].lower() in answer.lower(),
        })
    recall = sum(r['retrieval'] for r in results) / len(results)
    acc = sum(r['answer'] for r in results) / len(results)
    return recall, acc, results


recall, acc, details = evaluate(my_gold, k=3)
print(f'Recall@3:        {recall:.2f}')
print(f'Answer accuracy: {acc:.2f}')
print()
for r in details:
    print(f"{'OK' if r['retrieval'] else 'MISS'} retrieval | {'OK' if r['answer'] else 'MISS'} answer | {r['q']}")


## Step 6 - Share your finding

Prepare a one-minute summary for the group:

1. What documents did you use?
2. What question did you focus on?
3. Which technique did you apply?
4. Did the score improve, stay the same, or get worse? Why do you think that is?

There is no wrong answer. A technique that did not help is still a useful finding.

---

**What to explore next, on your own**

- Bonus A: see how LangChain and LlamaIndex implement the pipeline you just built.
- Bonus B: try hybrid search (BM25 + vector) on your documents.
- Bonus C: give the LLM a retrieval tool and let it decide when to search.
- RAGAS: set up automated evaluation that does not require a gold dataset.
- ChromaDB persistence: keep your collection between sessions and add documents over time.
